# 08 – SHAP : pourquoi le modèle prédit une panne ?

**Objectif :** ouvrir la « boîte noire » du meilleur classifieur (XGBoost, `03b_classification_synthetique`) et savoir **quelles variables déclenchent l'alerte** « panne dans les 48 h ».

**Principe de SHAP :** pour chaque prédiction, SHAP donne la **contribution de chaque variable** par rapport à une prédiction « moyenne ».
- valeur SHAP **positive** → la variable pousse vers « panne » ;
- valeur SHAP **négative** → elle pousse vers « normal » ;
- l'unité est le *log-odds* (échelle interne de XGBoost), pas une probabilité : seul le **signe** et l'**ordre de grandeur relatif** comptent.

**Entrées :** `data/processed/features_temporal.csv` et le modèle `models/classif_synth_xgboost.pkl` (créés par `03b`).
**Sorties :** `reports/figures/shap_*.png`, `reports/tables/shap_importance.csv`.

⚠️ SHAP explique **ce que le modèle a appris**, pas une cause physique. Et comme certaines variables sont corrélées (par exemple `vib_rms` et `vib_rms_mean_4`), leur importance se **partage** entre elles.

## 1. Chargement

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

warnings.filterwarnings("ignore")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "features.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from features import split_by_machine, SENSOR_COLS
(ROOT / "reports/figures").mkdir(parents=True, exist_ok=True)
(ROOT / "reports/tables").mkdir(parents=True, exist_ok=True)
SEED = 42

MODELE = "classif_synth_xgboost.pkl"        # pour expliquer le Random Forest : "classif_synth_random_forest.pkl"
chemin = ROOT / "models" / MODELE
if not chemin.exists():
    raise FileNotFoundError(f"{chemin} introuvable : lance d'abord 03b_classification_synthetique.ipynb.")
modele = joblib.load(chemin)

df = pd.read_csv(ROOT / "data/processed/features_temporal.csv")
FEATURES = [c for c in df.columns if any(c.startswith(s) for s in SENSOR_COLS)]   # mêmes variables que 03b
_, test = split_by_machine(df, test_size=0.25, seed=SEED)                         # mêmes machines de test que 03b
X_test, y_test = test[FEATURES], test["failure_within_h"]
print(MODELE, "|", len(FEATURES), "variables |", len(test), "lignes de test,", test.machine_id.nunique(), "machines")

ModuleNotFoundError: No module named 'shap'

## 2. Calcul des valeurs SHAP

On les calcule sur les machines de **test** (jamais vues à l'entraînement). `TreeExplainer` est la version rapide de SHAP pour les modèles à base d'arbres.

In [ ]:
explainer = shap.TreeExplainer(modele)
sv = explainer.shap_values(X_test)

# selon le modèle, shap renvoie un tableau (n, p), une liste [classe 0, classe 1] ou un tableau (n, p, 2) : on garde la classe « panne »
if isinstance(sv, list):
    sv = sv[1]
sv = np.asarray(sv)
if sv.ndim == 3:
    sv = sv[:, :, 1]
base = float(np.ravel(explainer.expected_value)[-1])
print("valeurs SHAP :", sv.shape, "| valeur de base :", round(base, 3))

## 3. Importance globale

Importance d'une variable = **moyenne de |SHAP|** sur toutes les mesures : plus elle est grande, plus la variable pèse dans les décisions du modèle.

In [ ]:
imp = pd.Series(np.abs(sv).mean(axis=0), index=FEATURES).sort_values(ascending=False)
imp.rename("mean_abs_shap").to_csv(ROOT / "reports/tables/shap_importance.csv")

top = imp.head(15)[::-1]
top.plot(kind="barh", figsize=(7, 5.5), color="steelblue")
plt.xlabel("moyenne de |SHAP|")
plt.title("15 variables les plus importantes")
plt.tight_layout()
plt.savefig(ROOT / "reports/figures/shap_importance.png", dpi=150)
plt.show()
imp.head(10).round(3)

## 4. Dans quel sens chaque variable agit ? (graphique en essaim)

Chaque point est une mesure. **Position horizontale** = effet sur la prédiction (à droite : vers « panne »). **Couleur** = valeur de la variable (rouge = élevée, bleu = faible). Lecture : si les points rouges sont à droite, une valeur élevée de la variable annonce une panne.

On trace un échantillon aléatoire de 5 000 mesures pour que le graphique reste lisible.

In [ ]:
rng = np.random.default_rng(SEED)
pos = rng.choice(len(X_test), size=min(5000, len(X_test)), replace=False)
plt.figure()
shap.summary_plot(sv[pos], X_test.iloc[pos], max_display=15, show=False)
plt.tight_layout()
plt.savefig(ROOT / "reports/figures/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Par capteur et par type de feature

Pour le rapport, on regroupe les 30 variables : **quel capteur** pèse le plus, et est-ce la **mesure brute** ou la **feature d'évolution** (moyenne, écart-type, pente, ratio à la baseline) qui aide le plus ?

In [ ]:
def capteur(f):
    return next(s for s in SENSOR_COLS if f.startswith(s))

def type_feature(f):
    for suf, nom in [("_mean_4", "moyenne 24 h"), ("_std_4", "écart-type 24 h"),
                     ("_slope_12", "pente 72 h"), ("_ratio_baseline", "ratio à la baseline")]:
        if f.endswith(suf):
            return nom
    return "mesure brute"

par_capteur = imp.groupby(capteur).sum().sort_values(ascending=False)
par_type = imp.groupby(type_feature).sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
(par_capteur[::-1] / imp.sum() * 100).plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Part de l'importance par capteur"); axes[0].set_xlabel("%")
(par_type[::-1] / imp.sum() * 100).plot(kind="barh", ax=axes[1], color="darkorange")
axes[1].set_title("Part de l'importance par type de feature"); axes[1].set_xlabel("%")
plt.tight_layout()
plt.savefig(ROOT / "reports/figures/shap_par_capteur_et_type.png", dpi=150)
plt.show()
print((par_capteur / imp.sum() * 100).round(1).to_string())
print()
print((par_type / imp.sum() * 100).round(1).to_string())

## 6. Relation entre la variable la plus importante et son effet

Graphique de dépendance : pour chaque valeur de la variable (axe horizontal), quel est son effet SHAP (axe vertical) ? On voit à partir de quelle valeur la variable commence à annoncer une panne.

In [ ]:
v1 = imp.index[0]
plt.figure()
shap.dependence_plot(v1, sv[pos], X_test.iloc[pos], show=False)
plt.tight_layout()
plt.savefig(ROOT / "reports/figures/shap_dependance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Variable tracée :", v1)

## 7. Explication d'**une** alerte

Une mesure d'une machine de test, environ **24 h avant la panne** : quelles variables ont poussé le modèle vers « panne » ? Barres rouges = vers la panne, bleues = vers « normal ». `E[f(X)]` est la valeur de base, `f(x)` la prédiction finale pour cette mesure.

In [ ]:
pannes = test[test.failure_within_h == 1]
ligne = pannes.iloc[(pannes.rul_h - 24).abs().argsort().iloc[0]]
i = test.index.get_loc(ligne.name)
print(f"Machine {ligne.machine_id}, cycle {int(ligne.cycle_id)} : panne dans {ligne.rul_h:.0f} h")

expl = shap.Explanation(values=sv[i], base_values=base, data=X_test.iloc[i].values, feature_names=FEATURES)
plt.figure()
shap.plots.waterfall(expl, max_display=12, show=False)
plt.tight_layout()
plt.savefig(ROOT / "reports/figures/shap_exemple_alerte.png", dpi=150, bbox_inches="tight")
plt.show()

## À retenir pour le rapport

- Compléter ici avec **les 3 variables les plus importantes** (§3), **le capteur dominant** et **le type de feature dominant** (§5), et la **valeur seuil** à partir de laquelle la variable principale annonce une panne (§6).
- Si les features d'évolution (ratio à la baseline, moyenne, pente) pèsent plus que les mesures brutes, c'est un argument pour le **feature engineering temporel** (`02_features`).
- Rappel : les données sont **synthétiques**. La dégradation est simulée (`hp = (âge/durée de vie)³`) et agit surtout sur les vibrations, la température, la pression et le courant : ce que SHAP retrouve ici est **ce qu'on a mis dans le simulateur**. Sur de vraies machines, l'importance des capteurs serait à confirmer.
- SHAP montre ce que le modèle utilise, pas une cause physique prouvée.
- Prochaine étape : **seuil de coût** (même `evaluate()` et hypothèses que `07_etude_cout`).